In [196]:
import sys
import os

sys.path.append(os.path.abspath(".."))

In [197]:
import pandas as pd
import plotly.graph_objects as go
from utils import DATA_FOLDER_PATH, describe_data, get_frequency_table, plot_df_chart

In [198]:
DICT = {
    "FEDFUNDS": {
        "title": "Federal Funds Effective Rate",
        "unit": "Percent"
    },
    "M2SL": {
        "title": "M2 ",
        "unit": "Billions of Dollars"
    },
    "WALCL": {
        "title": "Assets: Total Assets: Total Assets (Less Eliminations from Consolidation): Wednesday Level",
        "unit": "Millions of U.S. Dollars"
    },
    "GFDEBTN": {
        "title": "Federal Debt: Total Public Debt",
        "unit": "Millions of U.S. Dollars"
    },
    "WGS10YR": {
        "title": "Market Yield on U.S. Treasury Securities at 10-Year Constant Maturity, Quoted on an Investment Basis",
        "unit": "Percent"
    },
    "T10Y2Y": {
        "title": "10-Year Treasury Constant Maturity Minus 2-Year Treasury Constant Maturity",
        "unit": "Percent"
    },
    "FYFR": {
        "title": "Federal Receipts",
        "unit": "Millions of Dollars"
    },
    "FYONET": {
        "title": "Federal Net Outlays",
        "unit": "Millions of Dollars"
    },
}

In [199]:
INDICATOR = 'FYONET'

MoM_col = 'MoM ^ %'
YoY_col = 'YoY ^ %'

COL = YoY_col

In [200]:
df = pd.read_csv(os.path.join(DATA_FOLDER_PATH, INDICATOR + '.csv'))

df['DATE'] = pd.to_datetime(df['DATE'])
df.set_index('DATE', inplace=True)

# df[INDICATOR] = pd.to_numeric(df[INDICATOR])

# df[COL] = df[INDICATOR].pct_change(fill_method=None, periods=12 if COL == YoY_col else 1) * 100
df[COL] = df[INDICATOR].ffill().pct_change(periods=12 if COL == YoY_col else 1) * 100

# df[COL] = pd.to_numeric(df[COL])

# df.dropna(inplace=True)

df.tail()

,FYONET,YoY ^ %
DATE,,
2021-09-30,6822461,93.947909
2022-09-30,6273259,81.461257
2023-09-30,6134526,70.258544
2024-09-30,6734896,90.976200
2025-09-30,7009974,102.900592


- **Plot data**

In [201]:
plot_df_chart(
    df[[INDICATOR]],
    chart_title = DICT[INDICATOR]["title"],
    yaxis_title = DICT[INDICATOR]["unit"],
    use_markers=False,
    fill=True
)

- **% change**

In [202]:
fig = plot_df_chart(
    df[[COL]],
    chart_title = DICT[INDICATOR]["title"] + f' ({COL})',
    chart_type = "bar",
    yaxis_title = "%",
    draw=False,
)

AVG_PERIOD = 12 * 5

fig.add_trace(go.Scatter(
    x=df.index,
    y=df[COL].rolling(window=AVG_PERIOD).mean(),
    name='Average ' + COL + f' ({AVG_PERIOD})',
    line=dict(
        width=1,
        color="#F2C14E",
        # dash="dash"
    ),
    # hoverinfo="skip",
    # showlegend=False,
))

fig.show()

- **Stats**

In [203]:
df_stats, stats = describe_data(df[COL].dropna())

df_stats

,Value
Metric,
nobs,113
Min %,-80.657546
Max %,3093.955095
Mean %,269.086443
Median %,137.4237
Mode %,-80.657546
Variance,215432.291135
Skewness,3.921541
Kurtosis,16.603534


- **Freq**

In [204]:
freq_table = get_frequency_table(df.dropna()[COL])

freq_table

,Frequency,Probability %,Cumulative Probability %
Less than 448.44%,101,89.380531,89.380531
448.44% to 977.55%,7,6.194690,95.575221
977.55% to 1506.65%,0,0.000000,95.575221
1506.65% to 2035.75%,2,1.769912,97.345133
2035.75% to 2564.85%,2,1.769912,99.115044
Greater than 2564.85%,1,0.884956,100.000000


In [205]:
fig = plot_df_chart(
    freq_table[['Probability %']],
    chart_title = f'{DICT[INDICATOR]['title']} (% change frequency)',
    chart_type = "bar",
    yaxis_title = "Probability %",
    width=1000,
    height=700,
    show_rangeslider=False
)